In [ ]:
import pandas as pd
import numpy as np

# 1. Carrega o arquivo pulando as linhas de cabeçalho do INEP
df_raw = pd.read_csv('https://drive.google.com/uc?id=1D8noT18W2hq-c-IRcJRTFBZBq2ZBiXPX', skiprows=8, low_memory=False)

# Organiza os nomes das colunas de identificação
df_raw.columns = [
    'ano', 'regiao', 'uf', 'codigo_municipio', 'nome_municipio',
    'codigo_escola', 'nome_escola', 'localizacao', 'dependencia_adm'
] + list(df_raw.columns[9:])

# ==============================================================================
# 4. TRATAMENTO DE DADOS (Limpeza e Transformação)
# ==============================================================================
total_escolas_antes = len(df_raw)

# Mapeia as colunas de Reprovação (2_CAT) e Abandono/Evasão (3_CAT)
colunas_identificacao = ['uf', 'codigo_escola', 'nome_escola', 'localizacao', 'dependencia_adm']
colunas_metricas = [c for c in df_raw.columns if str(c).startswith('2_') or str(c).startswith('3_')]
df_combinado = df_raw[colunas_identificacao + colunas_metricas].copy()

# Renomeia os TOTAIS de cada bloco
df_combinado = df_combinado.rename(columns={
    '2_CAT_FUN': 'reprovacao_fundamental_total',
    '2_CAT_MED': 'reprovacao_medio_total',
    '3_CAT_FUN': 'evasao_fundamental_total',
    '3_CAT_MED': 'evasao_medio_total'
})

# Limpa a sujeira ('--') e converte as taxas para números reais (float)
colunas_para_converter = [
    'reprovacao_fundamental_total', 'reprovacao_medio_total',
    'evasao_fundamental_total', 'evasao_medio_total'
]
for col in colunas_para_converter:
    df_combinado[col] = pd.to_numeric(df_combinado[col].replace('--', np.nan), errors='coerce')

# Limpa o ID da escola (remove linhas com código nulo e converte para inteiro)
df_combinado = df_combinado.dropna(subset=['codigo_escola'])
df_combinado['codigo_escola'] = df_combinado['codigo_escola'].astype(int)

# Junta as dependências administrativas em "PÚBLICA"
mapeamento_publico = {
    'Federal': 'Pública',
    'Estadual': 'Pública',
    'Municipal': 'Pública',
    'Privada': 'Privada'
}
df_combinado['dependencia_adm'] = df_combinado['dependencia_adm'].replace(mapeamento_publico)

# Isola a tabela final exatamente com o que a equipe quer
tabela_equipe = df_combinado[[
    'uf', 'codigo_escola', 'nome_escola', 'dependencia_adm',
    'reprovacao_fundamental_total', 'reprovacao_medio_total',
    'evasao_fundamental_total', 'evasao_medio_total'
]].reset_index(drop=True)

linhas_excluidas = total_escolas_antes - len(tabela_equipe)
total_escolas_depois = len(tabela_equipe)

# ==============================================================================
# 5. RELATÓRIO DE TRATAMENTO DE DADOS
# ==============================================================================
print("\n" + "="*50)
print("RELATÓRIO DE TRATAMENTO DE DADOS")
print("="*50)
print(f"Total de escolas registradas (antes da limpeza): {total_escolas_antes}")
print(f"Total de linhas excluídas (sem código de escola): {linhas_excluidas}")
print(f"Total de escolas registradas (após a limpeza):  {total_escolas_depois}")
print("="*50 + "\n")

# ==============================================================================
# 6. VALIDAÇÃO VISUAL DOS DADOS
# ==============================================================================
display(tabela_equipe.head(100))

# ==============================================================================
# 7. EXPORTAÇÃO DO ARQUIVO FINAL
# ==============================================================================
tabela_equipe.to_csv('Censo_evasao_reprovacao_Limpa_2024.csv', index=False, sep=';', encoding='utf-8-sig')
print(f"Tabela exportada com sucesso! Arquivo salvo como: {nome_arquivo}")